# Handle Data Storage

### Create a list of all Zettel

In [ ]:
from zettelsortierung.DataTypes import *
from zettelsortierung.Transformation import *
from zettelsortierung.RegionDetection import OpenCVRegionDetector
from zettelsortierung.OCR import OCR, OCRpreprocessing, OCRpostprocessing
from zettelsortierung.DataModel import DataBase

from dotenv import load_dotenv
load_dotenv()

True

In [ ]:
connection_string = "sqlite:////home/jan/Dokumente/IT-Beratung Raring/Zettelsortierung/data/interim/pre-zettel.db"
db = DataBase(connection_string, echo=True)

2026-02-19 21:55:09,821 INFO sqlalchemy.engine.Engine BEGIN (implicit)
2026-02-19 21:55:09,823 INFO sqlalchemy.engine.Engine PRAGMA main.table_info("scans")
2026-02-19 21:55:09,825 INFO sqlalchemy.engine.Engine [raw sql] ()
2026-02-19 21:55:09,827 INFO sqlalchemy.engine.Engine PRAGMA temp.table_info("scans")
2026-02-19 21:55:09,828 INFO sqlalchemy.engine.Engine [raw sql] ()
2026-02-19 21:55:09,830 INFO sqlalchemy.engine.Engine PRAGMA main.table_info("zettel")
2026-02-19 21:55:09,831 INFO sqlalchemy.engine.Engine [raw sql] ()
2026-02-19 21:55:09,833 INFO sqlalchemy.engine.Engine PRAGMA temp.table_info("zettel")
2026-02-19 21:55:09,834 INFO sqlalchemy.engine.Engine [raw sql] ()
2026-02-19 21:55:09,835 INFO sqlalchemy.engine.Engine PRAGMA main.table_info("bounding_boxes")
2026-02-19 21:55:09,837 INFO sqlalchemy.engine.Engine [raw sql] ()
2026-02-19 21:55:09,838 INFO sqlalchemy.engine.Engine PRAGMA temp.table_info("bounding_boxes")
2026-02-19 21:55:09,839 INFO sqlalchemy.engine.Engine [raw

2026-02-19 21:55:09,844 INFO sqlalchemy.engine.Engine PRAGMA temp.table_info("ocr_results")
2026-02-19 21:55:09,845 INFO sqlalchemy.engine.Engine [raw sql] ()
2026-02-19 21:55:09,848 INFO sqlalchemy.engine.Engine 
CREATE TABLE scans (
	id VARCHAR NOT NULL, 
	file_name VARCHAR, 
	relative_path VARCHAR, 
	full_path VARCHAR, 
	PRIMARY KEY (id)
)


2026-02-19 21:55:09,849 INFO sqlalchemy.engine.Engine [no key 0.00104s] ()
2026-02-19 21:55:09,855 INFO sqlalchemy.engine.Engine 
CREATE TABLE zettel (
	id VARCHAR NOT NULL, 
	recto VARCHAR, 
	verso VARCHAR, 
	PRIMARY KEY (id), 
	FOREIGN KEY(recto) REFERENCES scans (id), 
	FOREIGN KEY(verso) REFERENCES scans (id)
)


2026-02-19 21:55:09,856 INFO sqlalchemy.engine.Engine [no key 0.00155s] ()
2026-02-19 21:55:09,862 INFO sqlalchemy.engine.Engine 
CREATE TABLE bounding_boxes (
	scan_id VARCHAR NOT NULL, 
	feature_id INTEGER NOT NULL, 
	x INTEGER, 
	y INTEGER, 
	w INTEGER, 
	h INTEGER, 
	PRIMARY KEY (scan_id, feature_id), 
	FOREIGN KEY(scan_id) REFE

In [ ]:
root = '../data/processed/Scans.parquet'
sammlung = Zettelsammlung.from_parquet(root)

In [4]:
#db.add_scan_list(scans)

In [5]:
detector = OpenCVRegionDetector()

pipeline = \
Composition(
    Batch(10000),
    SequentialApp(
        ParallelApp(
            PatchDetect(detector)
        ),
        Flatten(),
        ParallelApp(
            BoundingBoxPad(10),
        ),
        ToDataBase(db_adder=db.add_boundingbox_probe),
        ParallelApp(
            CutOutPatch(),
            ResizePatch(),
            BGR2RGB()
        ),
        Sort(key=lambda item: item.feature.shape[1]),
        Batch(16),
        ParallelApp(OCRpreprocessing()),
        SequentialApp(OCR()),
        ParallelApp(OCRpostprocessing()),
        Flatten(),
        ToDataBase(db_adder=db.add_ocr_probe)
    ),
    Flatten()
)

In [6]:
res = pipeline(sammlung)

2026-02-19 21:55:17,579 INFO sqlalchemy.engine.Engine BEGIN (implicit)
2026-02-19 21:55:17,586 INFO sqlalchemy.engine.Engine INSERT INTO bounding_boxes (scan_id, feature_id, x, y, w, h) VALUES (?, ?, ?, ?, ?, ?)
2026-02-19 21:55:17,587 INFO sqlalchemy.engine.Engine [generated in 0.00210s] [('A01-00000077_2', 0, 1489, 1131, 295, 78), ('A01-00000055_1', 0, 1339, 949, 206, 58), ('A01-00000059_1', 0, 1312, 991, 406, 175), ('A01-00000059_1', 1, 896, 1055, 279, 134), ('A01-00000077_1', 0, 1359, 393, 330, 100), ('A01-00000043_1', 0, 1088, 99, 108, 91), ('A01-00000043_1', 1, 1263, 724, 122, 95), ('A01-00000043_1', 2, 1171, 809, 508, 157)  ... displaying 10 of 144 total bound parameter sets ...  ('A01-00000011_1', 1, 1452, 1155, 331, 92), ('A01-00000011_1', 0, 44, 744, 506, 145)]
2026-02-19 21:55:17,592 INFO sqlalchemy.engine.Engine COMMIT
144
2026-02-19 21:55:23,751 INFO sqlalchemy.engine.Engine BEGIN (implicit)
2026-02-19 21:55:23,758 INFO sqlalchemy.engine.Engine INSERT INTO ocr_results (sca